In [ ]:
from scripts import feature_extraction as fe
import numpy as np, pandas as pd
import os, random

os.makedirs("data/features", exist_ok=True)
csv_file = "data/farseeing/F_00002186-01-2013-11-15-13-50-18.csv"
fs_metadata = 100   # replace with metadata value

features, timestamps = fe.process_farseeing_file(csv_file, fs_metadata)

np.save("data/features/features_subject01.npy", features)
np.save("data/features/timestamps_subject01.npy", timestamps)

print("Saved feature matrix and timestamps.")

Processing: data/farseeing/F_00002186-01-2013-11-15-13-50-18.csv
Original FS: 100 Hz
Total windows: 1198
Saved feature matrix and timestamps.


In [2]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd

from scripts.feature_extraction import process_farseeing_file


ADL_DIR = "data/farseeing/adl"
FALL_DIR = "data/farseeing/fall"

OUTPUT_DIR = "features/w3_s1"
INDEX_FILE = "features/feature_index_w3_s1.csv"
meta = pd.read_excel("data/farseeing/description.xlsx", engine="openpyxl")

os.makedirs(OUTPUT_DIR, exist_ok=True)

N_ADL = 30
N_FALL = 30

In [3]:
def parse_fall_filename(filename):
    """
    Example:
    F_00002186-01-2013-11-15-13-50-18.csv
    """
    base = filename.replace(".csv", "")
    subject_id = base.split("_")[1].split("-")[0]
    file_id = "-".join(base.split("-")[2:])  # keep session timestamp
    return subject_id, file_id

def parse_adl_filename(filename):
    """
    Example:
    subAtaxia1_1.csv
    """
    base = filename.replace(".csv", "")
    subject_id, file_num = base.split("_")
    file_id = file_num
    return subject_id, file_id

def get_sampling_rate(filename):
    """
    Example:
    F_00002186-01-2013-11-15-13-50-18.csv
    file id is 00002186-01
    """
    file_id = filename.split("_")[1].split("-")[0] + "-" + filename.split("_")[1].split("-")[1]
    fs = meta[meta["Randomnumber"] == file_id]["Sample_rate_Hz"].iloc[0]
    return fs

In [4]:
adl_files = [f for f in os.listdir(ADL_DIR) if f.endswith(".csv")]
fall_files = [f for f in os.listdir(FALL_DIR) if f.endswith(".csv")]

random.seed(42)

adl_sample = random.sample(adl_files, min(N_ADL, len(adl_files)))
fall_sample = random.sample(fall_files, min(N_FALL, len(fall_files)))

selected_files = [("adl", f) for f in adl_sample] + [("fall", f) for f in fall_sample]

print(len(selected_files), "files selected")

60 files selected


In [5]:
if os.path.exists(INDEX_FILE):
    index_df = pd.read_csv(INDEX_FILE)
else:
    index_df = pd.DataFrame(columns=[
        "subject_id",
        "file_id",
        "label_type",
        "feature_path",
        "timestamp_path"
    ])

In [6]:
for label_type, filename in selected_files:

    try:
        if label_type == "adl":
            subject_id, file_id = parse_adl_filename(filename)
            csv_path = os.path.join(ADL_DIR, filename)
        else:
            subject_id, file_id = parse_fall_filename(filename)
            csv_path = os.path.join(FALL_DIR, filename)

        if filename.startswith("F_"):
            fs = get_sampling_rate(filename)
        else:
            fs = 100

        features, timestamps = process_farseeing_file(csv_path, fs)

        feature_name = f"{subject_id}__{file_id}.npy"
        timestamp_name = f"{subject_id}__{file_id}_ts.npy"

        feature_path = os.path.join(OUTPUT_DIR, feature_name)
        timestamp_path = os.path.join(OUTPUT_DIR, timestamp_name)

        np.save(feature_path, features)
        np.save(timestamp_path, timestamps)

        # update index
        index_df.loc[len(index_df)] = [
            subject_id,
            file_id,
            label_type,
            feature_path,
            timestamp_path
        ]

        print(f"Saved: {subject_id} | {file_id}")

    except Exception as e:
        print(f"Error processing {filename}: {e}")

Processing: data/farseeing/adl/subPSP4_7.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subPSP4 | 7
Processing: data/farseeing/adl/subRehab5_2.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab5 | 2
Processing: data/farseeing/adl/subRehab3_2.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab3 | 2
Processing: data/farseeing/adl/subRehab1_10.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab1 | 10
Processing: data/farseeing/adl/subRehab2_6.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab2 | 6
Processing: data/farseeing/adl/subRehab4_4.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab4 | 4
Processing: data/farseeing/adl/subRehab5_1.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab5 | 1
Processing: data/farseeing/adl/subRehab1_6.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab1 | 6
Processing: data/farseeing/adl/subRehab2_12.csv
Original FS: 100 Hz
Total windows: 3597
Saved: subRehab2 | 12
Processing: data/farseeing/a

In [7]:
index_df.to_csv(INDEX_FILE, index=False)

print("Feature extraction batch complete.")

Feature extraction batch complete.
